# 04 - Degradation model: selection, implementation and validation (workflow step 4)

**Adopted model (published, not fitted here):** Xu et al., *IEEE Trans. Smart Grid* 9(2), 2018 --
cycle-depth stress $\Phi(\delta)=(k_{\delta1}\delta^{k_{\delta2}}+k_{\delta3})^{-1}$, SOC stress
$S_\sigma(\sigma)=e^{k_\sigma(\sigma-\sigma_{ref})}$, calendar stress $k_t t S_\sigma$, and the
nonlinear capacity fade with SEI formation $D=L(F)=1-\alpha_{sei}e^{-\beta_{sei}F}-(1-\alpha_{sei})e^{-F}$,
$SOH=1-D$. For the EMS, the convex piecewise-linear cycle-depth representation of Xu et al.,
*IEEE Trans. Power Syst.* 33(2), 2018 is used.

Online increment (envelope, EMS, SOH state):
$\Delta F_k^{cyc}=\tfrac12|G(\delta_{k+1})-G(\delta_k)|$, $\delta=SOC_{max}-SOC$,
$G(\delta)=\Phi(\delta)S_\sigma(SOC_{max}-\delta/2)$;
$\Delta D_k = L(F_k+\Delta F_k^{cyc}+\Delta F_k^{cal})-L(F_k)$.
Ex-post SOH is always evaluated with full rainflow counting.

Outputs: `output/results/results_degradation_validation.csv`, `output/results/results_degradation_parameters.csv`,
`output/experiment_figures/Fig_degradation_model.pdf` (supplementary).

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from dfm_acdc.config import RESULTS_DIR, FIG_DIR, BESSParams, DegradationParams, DFMParams
from dfm_acdc.degradation import XuDegradation
from dfm_acdc.envelope import degradation_allowance
from dfm_acdc import plotstyle

plotstyle.apply_style()
b, dp, dfm = BESSParams(), DegradationParams(), DFMParams()
deg = XuDegradation(dp, b.soc_max, b.soc_min)
tests = []
def check(name, value, ok, tol=""):
    tests.append({"test": name, "value": value, "tolerance": tol, "pass": bool(ok)})
    print(f"[{'PASS' if ok else 'FAIL'}] {name}: {value}")

## Rainflow counting -- ASTM E1049 reference example

In [ ]:
cyc = deg.rainflow([-2, 1, -3, 5, -1, 3, -4, 4, -2])
got = sorted((float(r), n) for r, m, n in cyc)
ref = sorted([(3, .5), (4, .5), (4, 1.0), (6, .5), (8, .5), (8, .5), (9, .5)])
check("ASTM E1049 example: ranges and counts", got, got == [(float(r), n) for r, n in ref])

## Online increment vs. rainflow

In [ ]:
for dod in (0.1, 0.3, 0.5, 0.8):
    soc = np.r_[np.linspace(b.soc_max, b.soc_max - dod, 25), np.linspace(b.soc_max - dod, b.soc_max, 25)[1:]]
    on = sum(deg.dF_cycle(soc[i], soc[i + 1]) for i in range(len(soc) - 1))
    rf, _, _ = deg.F_rainflow(soc, 0.25)
    check(f"full cycle from SOC_max, depth {dod}: online = rainflow = G(delta)", f"{on:.4e} / {rf:.4e} / {float(deg.G(dod)):.4e}",
          abs(on - rf) / rf < 1e-10 and abs(rf - deg.G(dod)) / rf < 1e-10, "1e-10 rel")
rng = np.random.default_rng(3)
ratios = []
for _ in range(200):
    steps = rng.normal(0, 0.05, 96)
    soc = np.clip(0.5 + np.cumsum(steps), b.soc_min, b.soc_max)
    on = sum(deg.dF_cycle(soc[i], soc[i + 1]) for i in range(95))
    rf, _, _ = deg.F_rainflow(soc, 0.25)
    if rf > 0:
        ratios.append(on / rf)
ratios = np.array(ratios)
print(f"random SOC profiles: online/rainflow mean {ratios.mean():.3f}, P5 {np.percentile(ratios,5):.3f}, P95 {np.percentile(ratios,95):.3f}")
pd.DataFrame({"online_over_rainflow": ratios}).describe(percentiles=[.05, .5, .95]).T.to_csv(
    os.path.join(RESULTS_DIR, "results_degradation_online_vs_rainflow_random.csv"))

**Characterization (not a pass/fail test).** The online increment is exact for cycles starting
at the upper SOC limit, but for strongly irregular profiles with many nested micro-cycles (the
random walks above, much choppier than any EMS dispatch) it deviates from rainflow, mostly on
the conservative side (it ignores nesting). This is why every reported $\Delta SOH$ is computed
ex post with rainflow, while the online estimate is used only inside the envelope and the EMS;
the agreement on the actual dispatch profiles is reported in notebook 07.

## Piecewise-linear segments for the EMS (convex hull)

In [ ]:
kappa, w, slopes = deg.segment_costs()
seg = pd.DataFrame({"segment": np.arange(1, len(kappa) + 1), "depth_from": np.arange(len(kappa)) * w,
                    "depth_to": np.arange(1, len(kappa) + 1) * w, "raw_slope": slopes, "hull_slope": kappa})
print(seg.round(8).to_string(index=False))
check("hull slopes non-decreasing (convex)", "", np.all(np.diff(kappa) >= -1e-15))
check("hull preserves total full-depth stress", f"{np.sum(kappa)*w:.6e} vs {float(deg.G(b.soc_max-b.soc_min)):.6e}",
      abs(np.sum(kappa) * w - deg.G(b.soc_max - b.soc_min)) < 1e-15)

## SOH mapping, degradation allowance and usable energy

In [ ]:
rows = []
for soh in (1.0, 0.95, 0.90, 0.85):
    F = deg.F_from_soh(soh)
    D_allow, D_max, _ = degradation_allowance(deg, F, 0.0, dfm)
    rows.append({"SOH_0": soh, "F_0": F, "dL_dF": float(deg.dL(F)), "E_use_kWh": soh * b.E_rated_kwh,
                 "calendar_dD_per_day": float(deg.dL(F) * deg.dF_calendar(0.5, 24.0)),
                 "D_max_cyc_per_day": D_max, "ref_cycle_F": deg.reference_cycle_F(dfm.DoD_ref)})
par = pd.DataFrame(rows)
par.to_csv(os.path.join(RESULTS_DIR, "results_degradation_parameters.csv"), index=False)
print(par.to_string(index=False))
check("L(F_from_soh(SOH)) = 1 - SOH", "", all(abs(deg.L(deg.F_from_soh(s)) - (1 - s)) < 1e-9 for s in (0.99, 0.9, 0.8)))

In [ ]:
val = pd.DataFrame(tests)
val.to_csv(os.path.join(RESULTS_DIR, "results_degradation_validation.csv"), index=False)
print(f"\n{val['pass'].sum()}/{len(val)} tests passed")
assert val["pass"].all()

## Supplementary figure -- adopted degradation model

In [ ]:
FIG_WIDTH, FIG_HEIGHT = plotstyle.IEEE_PAGE_WIDTH_IN, 2.4
fig, axs = plt.subplots(1, 3, figsize=(FIG_WIDTH, FIG_HEIGHT))
d = np.linspace(0.005, b.soc_max - b.soc_min, 200)
ax = axs[0]
ax.plot(d, deg.phi(d) * 1e5, color=plotstyle.CAT["blue"], label=r"$\Phi(\delta)$")
ax.plot(d, deg.G(d) * 1e5, color=plotstyle.CAT["orange"], label=r"$G(\delta)$")
ax.set_xlabel(r"Cycle depth $\delta$"); ax.set_ylabel(r"Stress per cycle [$10^{-5}$]")
ax.legend(frameon=False, fontsize=6.5)
ax = axs[1]
x = np.arange(len(kappa))
ax.bar(x - 0.2, slopes * 1e5, width=0.4, color=plotstyle.INK_MUTED, label="exact")
ax.bar(x + 0.2, kappa * 1e5, width=0.4, color=plotstyle.CAT["aqua"], label="convex hull (EMS)")
ax.set_xlabel("Depth segment"); ax.set_ylabel(r"Marginal stress [$10^{-5}$/SOC]")
ax.set_xticks(x); ax.set_xticklabels([str(i + 1) for i in x]); ax.legend(frameon=False, fontsize=6.5)
ax = axs[2]
F = np.linspace(0, 0.25, 300)
ax.plot(F, 1 - deg.L(F), color=plotstyle.CAT["violet"])
for soh in (1.0, 0.95, 0.90, 0.85):
    ax.plot(deg.F_from_soh(soh), soh, "o", ms=4, color=plotstyle.INK_PRIMARY)
ax.axhline(dp.soh_eol, color=plotstyle.LIMIT_COLOR, lw=0.8, ls="--", label="EOL")
ax.set_xlabel("Linearized degradation $F$"); ax.set_ylabel("SOH"); ax.legend(frameon=False, fontsize=6.5)
for a, lab in zip(axs, "abc"):
    a.text(-0.32, 1.04, f"({lab})", transform=a.transAxes, fontsize=8, va="top")
plotstyle.savefig(fig, os.path.join(FIG_DIR, "Fig_degradation_model.pdf"))
plt.show()

## Takeaway

The degradation model is a published LMO-cell model used with its published parameters. Its
rainflow implementation reproduces the ASTM reference example, the online increment equals
rainflow for cycles from the upper SOC limit, and the EMS segment costs are the convex hull of
the exact cycle-depth stress (the exact stress is slightly concave at very shallow depths). The
nonlinear SEI term makes a fresh battery ($SOH\approx1$) age several times faster per unit
stress than a mid-life one; the degradation allowance is therefore defined in stress units
($N_{ref}$ reference cycles per day) and converted to SOH units with $L'(F_k)$.